In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) CSV yükleme (Colab)
try:
    from google.colab import files
    uploaded = files.upload()  # kredi_onay.csv dosyanı yükle
    csv_path = next(iter(uploaded.keys()))
except Exception:
    csv_path = "kredi_onay.csv"

df0 = pd.read_csv(csv_path, encoding="ISO-8859-1", sep=";")
print("Shape:", df0.shape)
display(df0.head())

# 2) years_employed / credit_history_years temizliği (tarih benzeri -> NaN)
def clean_years_like(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int,float,np.integer,np.floating)):
        return float(x)
    s = str(x).strip()
    # harf içeriyorsa (Şub/Eyl vb.) süre olamaz -> NaN
    if re.search(r"[A-Za-zŞşĞğÜüÖöİı]", s):
        return np.nan
    try:
        return float(s.replace(",", "."))
    except:
        return np.nan

df = df0.copy()

for col in ["years_employed", "credit_history_years"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_years_like)

# interest_rate kesinlikle numerik olmalı
if "interest_rate" in df.columns:
    df["interest_rate"] = pd.to_numeric(df["interest_rate"].astype(str).str.replace(",", "."), errors="coerce")

# kategorikler
for c in ["occupation_status", "product_type", "loan_intent"]:
    if c in df.columns:
        df[c] = df[c].astype("category")

# numeric dönüşüm
expected_numeric = [
    "age","annual_income","credit_score","savings_assets","current_debt",
    "defaults_on_file","delinquencies_last_2yrs","derogatory_marks",
    "loan_amount","debt_to_income_ratio","loan_to_income_ratio",
    "payment_to_income_ratio","loan_status",
    "years_employed","credit_history_years","interest_rate"
]
for c in expected_numeric:
    if c in df.columns and df[c].dtype == object:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "."), errors="coerce")

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("\nSayısal sütunlar:", numeric_cols)

# 3) Konum & değişim ölçüleri
desc = df[numeric_cols].describe().T
desc["median"] = df[numeric_cols].median()
desc["var"] = df[numeric_cols].var()
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)
desc["IQR"] = q3 - q1
stats_table = desc[["count","mean","median","std","var","min","25%","50%","75%","IQR","max"]].round(4)

print("\n=== Konum & Değişim Tablosu ===")
display(stats_table)

print("""Açıklama:
- Konum ölçüleri: mean (ortalama), median (medyan) → tipik değer.
- Değişim ölçüleri: std (standart sapma), var (varyans), IQR (Q3-Q1) → verinin yayılımı.
- mean ile median çok farklıysa dağılım çarpık olabilir (uç değerler ortalamayı çekebilir).
""")

# 4) Boxplot + her birinin altına 1–2 cümle açıklama
def summary_stats(series: pd.Series):
    s = series.dropna().astype(float)
    q1 = s.quantile(0.25); q3 = s.quantile(0.75)
    return {
        "count": int(s.count()),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "std": float(s.std(ddof=1)),
        "var": float(s.var(ddof=1)),
        "min": float(s.min()),
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(q3-q1),
        "max": float(s.max()),
        "skew": float(s.skew())
    }

def iqr_outlier_info(series: pd.Series):
    st = summary_stats(series)
    q1, q3, iqr = st["q1"], st["q3"], st["iqr"]
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr
    s = series.dropna().astype(float)
    outliers = ((s < lower) | (s > upper)).sum()
    return lower, upper, int(outliers), float(outliers/len(s)) if len(s) else 0.0, st

candidate_continuous = [
    "age","years_employed","annual_income","credit_score","credit_history_years",
    "savings_assets","current_debt","loan_amount","interest_rate",
    "debt_to_income_ratio","loan_to_income_ratio","payment_to_income_ratio"
]
continuous_cols = [c for c in candidate_continuous if c in df.columns]

print("\n=== Boxplotlar + Alt açıklamalar ===")
for c in continuous_cols:
    lower, upper, outliers, out_pct, st = iqr_outlier_info(df[c])

    plt.figure()
    plt.boxplot(df[c].dropna().values, vert=True, showfliers=True)
    plt.title(f"Boxplot - {c}")
    plt.ylabel(c)
    plt.tight_layout()
    plt.show()

    skew = st["skew"]
    skew_txt = "sağa çarpık" if skew > 0.5 else ("sola çarpık" if skew < -0.5 else "yaklaşık simetrik")

    print(f"""Açıklama ({c}):
- Medyan={st['median']:.4f}, Ortalama={st['mean']:.4f}; dağılım {skew_txt} (çarpıklık≈{skew:.2f}).
- IQR kuralına göre aykırı değer adayı {outliers} gözlem (%{out_pct*100:.2f}); sınırlar ≈ [{lower:.4f}, {upper:.4f}].
""")

# 5) Korelasyon matrisi + altına açıklama
corr = df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr.values, aspect="auto")
plt.title("Korelasyon Matrisi (Pearson)")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr.index)), corr.index, fontsize=7)
plt.colorbar(label="corr")
plt.tight_layout()
plt.show()

pairs = []
cols = corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        pairs.append((cols[i], cols[j], float(corr.iloc[i, j])))

top_pairs = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)[:10]

print("""Korelasyon matrisi açıklaması:
- +1'e yakın: güçlü pozitif ilişki, -1'e yakın: güçlü negatif ilişki, 0'a yakın: zayıf doğrusal ilişki.
- Korelasyon nedensellik değildir; sadece birlikte değişimi ölçer.

En güçlü korelasyonlar (|r| büyük):
""")
for a, b, r in top_pairs:
    print(f"- {a} ↔ {b}: r = {r:.3f}")


FileNotFoundError: [Errno 2] No such file or directory: 'kredi_onay.csv'